In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter
import random
import sys
sys.path.insert(0, '/cosma/home/dp004/dc-zhan5')
import MyHaloPS as ps
import illustris_python as il


In [2]:
boxsize = 205
snapnum = 40

In [3]:
import seaborn as sns
palette_tab10 = sns.color_palette("colorblind", 10)

In [4]:
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["ytick.direction"] = "in"
plt.rc("font", family="STIXGeneral", size=18)
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams['figure.figsize'] = (2*10/3, 4)
plt.rcParams["legend.frameon"] = False


plt.rcParams["xtick.major.size"] = 5
plt.rcParams["ytick.major.size"] = 5
plt.rcParams["ytick.minor.visible"] = True
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["xtick.top"] = True
plt.rcParams["ytick.right"] = True
plt.rcParams["lines.linewidth"] = 2

In [5]:
basePath = '/cosma7/data/dp004/dc-zhan5/TNG300-1'

In [6]:
fields = ['GroupFirstSub', "GroupSFR", "GroupMass", "GroupNsubs", 
          "GroupPos", "GroupMassType", "GroupBHMass", "Group_M_TopHat200", "Group_M_Crit200"]
header = il.groupcat.loadHeader(f"{basePath}/output", snapnum)
halos = il.groupcat.loadHalos(f"{basePath}/output", snapnum, fields=fields)

In [7]:
fields = ["SubhaloSFR", # [Msun/yr]
          "SubhaloGrNr",
         "SubhaloFlag",
         "SubhaloPos",
         "SubhaloCM", "SubhaloHalfmassRad", "SubhaloHalfmassRadType", "SubhaloMass", "SubhaloBHMass",
         "SubhaloMassType", "SubhaloVmax", "SubhaloVmaxRad"] # [10^10 Msun/h]
print(len(fields))
subhalos = il.groupcat.loadSubhalos(f"{basePath}/output", snapnum, fields=fields)
print(type(subhalos))


12


<class 'dict'>


In [8]:
mvir = np.log10(halos["Group_M_TopHat200"]*1e10)

/tmp/ipykernel_2821613/565162213.py:1: RuntimeWarning: divide by zero encountered in log10
  mvir = np.log10(halos["Group_M_TopHat200"]*1e10)


In [9]:
group_sfrs = np.log10(halos["GroupSFR"])

/tmp/ipykernel_2821613/618587702.py:1: RuntimeWarning: divide by zero encountered in log10
  group_sfrs = np.log10(halos["GroupSFR"])


In [10]:
pos = halos["GroupPos"]/1e3

In [11]:
cent_mask_all = np.full(len(subhalos["SubhaloSFR"]), False)
cent_mask_all[halos["GroupFirstSub"][halos["GroupFirstSub"]>-1]] = True

In [12]:
cent_sfrs1 = subhalos["SubhaloSFR"][halos["GroupFirstSub"][halos["GroupFirstSub"]>-1]]
cent_sfrs = np.zeros(len(mvir))
cent_sfrs[halos["GroupFirstSub"]>-1] = cent_sfrs1
#cent_sfrs = np.log10(cent_sfrs)

In [13]:
cent_submass = np.zeros(len(mvir))
cent_submass[halos["GroupFirstSub"]>-1] = subhalos["SubhaloMass"][halos["GroupFirstSub"][halos["GroupFirstSub"]>-1]]
cent_submassx = np.log10(cent_submass*1e10)

/tmp/ipykernel_2821613/1109232107.py:3: RuntimeWarning: divide by zero encountered in log10
  cent_submassx = np.log10(cent_submass*1e10)


In [14]:
vmax = subhalos["SubhaloVmax"]
rmax = subhalos["SubhaloVmaxRad"]

In [15]:
cent_vmax1 = subhalos["SubhaloVmax"][halos["GroupFirstSub"][halos["GroupFirstSub"]>-1]]
cent_vmax = np.zeros(len(mvir))
cent_vmax[halos["GroupFirstSub"]>-1] = cent_vmax1

In [16]:
cent_rmax1 = subhalos["SubhaloVmaxRad"][halos["GroupFirstSub"][halos["GroupFirstSub"]>-1]]
cent_rmax = np.zeros(len(mvir))
cent_rmax[halos["GroupFirstSub"]>-1] = cent_rmax1

In [24]:
cent_cprox = cent_vmax/cent_rmax

/tmp/ipykernel_2821613/2104352924.py:1: RuntimeWarning: invalid value encountered in divide
  cent_cprox = cent_vmax/cent_rmax


In [32]:
mask_mass = (mvir > 11) & (mvir < 11.1)
mvir1 = mvir[mask_mass]
pos1 = pos[mask_mass]
cent_cprox1 = cent_cprox[mask_mass]

In [33]:
length =  np.sum(mask_mass) # Define the size of the array (must be even for equal halves)

In [31]:
with open("logM11-11.1/weight1.txt", "w") as f:
    for ihalo in range(np.sum(mask_mass)):
        print(mvir1[ihalo], pos1[:,0][ihalo], pos1[:,1][ihalo], pos1[:,2][ihalo], 1, file=f)

In [64]:
# Create an array with an equal number of 1s and 2s

num_ones = length // 2
num_twos = length - num_ones

# Create an array with half 1s and half 2s
array = np.array([1] * num_ones + [5] * num_twos)

np.random.seed(4)
# Shuffle the array randomly
np.random.shuffle(array)

In [65]:
with open("logM11-11.1/weight1_5_random_seed4.txt", "w") as f:
    for ihalo in range(np.sum(mask_mass)):
        print(mvir1[ihalo], pos1[:,0][ihalo], pos1[:,1][ihalo], pos1[:,2][ihalo], array[ihalo], file=f)

29.195023

In [57]:
med = np.median(cent_cprox1)
mask_low = cent_cprox1 < med
mask_high = cent_cprox1 > med
cprox_weight = np.ones(length)
cprox_weight[mask_high] = 5

In [54]:
with open("logM11-11.1/weight1_5_cprox.txt", "w") as f:
    for ihalo in range(np.sum(mask_mass)):
        print(mvir1[ihalo], pos1[:,0][ihalo], pos1[:,1][ihalo], pos1[:,2][ihalo], cprox_weight[ihalo], file=f)

In [62]:
med = np.median(cent_cprox1)
mask_low = cent_cprox1 < med
mask_high = cent_cprox1 > med
cprox_weight = np.ones(length)
cprox_weight[mask_low] = 5

In [63]:
with open("logM11-11.1/weight5_1_cprox.txt", "w") as f:
    for ihalo in range(np.sum(mask_mass)):
        print(mvir1[ihalo], pos1[:,0][ihalo], pos1[:,1][ihalo], pos1[:,2][ihalo], cprox_weight[ihalo], file=f)

In [60]:
cprox_weight = np.full(length, 3)
med = np.median(cent_cprox1)
per75 = np.percentile(cent_cprox1, [75])
per25 = np.percentile(cent_cprox1, [25])
mask_low = cent_cprox1 < per25
mask_high = cent_cprox1 > per75
cprox_weight = np.ones(length)
cprox_weight[mask_low] = 1
cprox_weight[mask_high] = 5

In [61]:
with open("logM11-11.1/weight1(25)_5(75)_cprox.txt", "w") as f:
    for ihalo in range(np.sum(mask_mass)):
        print(mvir1[ihalo], pos1[:,0][ihalo], pos1[:,1][ihalo], pos1[:,2][ihalo], cprox_weight[ihalo], file=f)